In [ ]:
# import langchain
# langchain.debug = True

from langchain.callbacks.base import BaseCallbackHandler
from langchain_ollama import ChatOllama


class PrintRequestHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("PROMPTS:", prompts)
        print("KWARGS:", kwargs)


llm = ChatOllama(
    model="gemma3:27b",
    temperature=0.1,
    callbacks=[PrintRequestHandler()]
)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

tagging_prompt = ChatPromptTemplate.from_template(
    """
    Extract the desired information from the following passage.

    Only extract the properties mentioned in the 'Classification' function.

    Passage:
    {input}
    """
)


class Classification(BaseModel):
    sentiment: str = Field(
        description="the sentiment of the text",
        enum=["happy", "neutral", "sad"]
    )
    aggressiveness: float = Field(
        description="describes how aggressive the statement is, the higher the number the more aggressive it is on a scale from 0 to 1"
    )
    spoken_language: str = Field(
        description="the language the text is written in",
        enum=["spanish", "english", "french", "german", "italian"]
    )


structured_llm = llm.with_structured_output(Classification, include_raw=True)

In [ ]:
inp = "Ya quisiera llegar a mi casa y comer mi plato favorito. Eso me haría muy feliz."
prompt = tagging_prompt.invoke({"input": inp})
response = structured_llm.invoke(prompt)

response["parsed"]